# 03 — Il confronto (Versione 4)

Tutti i modelli finiti della Versione 4 (TF-IDF con e senza titolo, ModernBERT titolo + articolo, con e senza i
criteri NewsGuard, con i loro seed) e, come **riferimenti**, quelli della Versione 2 sul solo articolo (stesso
dataset, stesso training, stessi fold, stesse 304 testate): `feature_tuned` e ModernBERT-base seed 42, 1, 2
(copiati in `../Results/riferimenti_v2/`).

Per testata: TF-IDF col voto e con la media delle probabilità, i transformer con la media. Per le configurazioni con
più seed l'**ensemble** (media delle probabilità). Intervalli al 95% con un **bootstrap per gruppo** e differenze con
il TF-IDF di riferimento (`feature_titolo_tuned` se c'è, altrimenti `v2_feature_tuned`). Metriche principali: F1
macro a 5 livelli e binario.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

from comune import CLASSI, RESULTS, VERSIONI, metriche_di, per_testata, per_testata_media

B = 2000
modelli = {}
for d in sorted(RESULTS.glob("feature*")):
    if (d / "FATTO").exists():
        modelli[d.name] = pd.read_csv(d / "articoli.csv")
for d in sorted((RESULTS / "transformer").glob("*")):
    if (d / "FATTO").exists():
        modelli[d.name] = pd.read_csv(d / "articoli.csv")
RIF_V2 = RESULTS.parent / "riferimenti_v2"
for f in sorted(RIF_V2.glob("*.csv")):
    modelli["v2_" + f.stem] = pd.read_csv(f)
seed = {}
for n in list(modelli):
    m = re.match(r"(.+)_s\d+$", n)
    if m:
        seed.setdefault(m[1], []).append(n)
for config, nomi in seed.items():
    if len(nomi) > 1:
        parti = [modelli[n].set_index("article_id") for n in nomi]
        e = parti[0][["domain", "fold", "y_true"]].copy(); e[CLASSI] = sum(p.loc[e.index, CLASSI] for p in parti) / len(parti)
        e["y_pred"] = e[CLASSI].to_numpy().argmax(1)
        modelli[f"{config}_ensemble{len(nomi)}"] = e.reset_index()
print("modelli:", list(modelli))
T, righe = {}, {}
for n, p in modelli.items():
    for agg, fun in ([("voto", per_testata), ("media", per_testata_media)] if n.startswith(("feature", "v2_feature")) else [("media", per_testata_media)]):
        t = fun(p); T[(n, agg)] = t
        righe[(n, agg)] = {k: v for k, v in metriche_di(t.y_true, t.y_pred).items() if not isinstance(v, list)}
tutto = pd.DataFrame(righe).T; tutto.index.names = ["modello", "aggregazione"]
tutto.to_csv(RESULTS / "03_confronto_tutte_le_metriche.csv")
tutto[["n", "f1_macro", "bin_accuratezza", "accuratezza", "tolerant_1"]].round(3)

modelli: ['feature_titolo', 'feature_titolo_tuned', 'v2_feature_tuned', 'v2_modernbert_solo_articolo_s1', 'v2_modernbert_solo_articolo_s2', 'v2_modernbert_solo_articolo_s42', 'v2_modernbert_solo_articolo_ensemble3']


n  f1_macro  \
modello                               aggregazione                    
feature_titolo                        voto          304.0     0.393   
                                      media         304.0     0.375   
feature_titolo_tuned                  voto          304.0     0.390   
                                      media         304.0     0.369   
v2_feature_tuned                      voto          304.0     0.370   
                                      media         304.0     0.363   
v2_modernbert_solo_articolo_s1        media         304.0     0.323   
v2_modernbert_solo_articolo_s2        media         304.0     0.356   
v2_modernbert_solo_articolo_s42       media         304.0     0.364   
v2_modernbert_solo_articolo_ensemble3 media         304.0     0.341   

                                                    bin_accuratezza  \
modello                               aggregazione                    
feature_titolo                        voto                    0.734   
                                      media                   0.737   
feature_titolo_tuned                  voto                    0.757   
                                      media                   0.724   
v2_feature_tuned                      voto                    0.773   
                                      media                   0.763   
v2_modernbert_solo_articolo_s1        media                   0.704   
v2_modernbert_solo_articolo_s2        media                   0.701   
v2_modernbert_solo_articolo_s42       media                   0.740   
v2_modernbert_solo_articolo_ensemble3 media                   0.707   

                                                    accuratezza  tolerant_1  
modello                               aggregazione                           
feature_titolo                        voto                0.454       0.757  
                                      media               0.444       0.757  
feature_titolo_tuned                  voto                0.424       0.780  
                                      media               0.395       0.753  
v2_feature_tuned                      voto                0.431       0.799  
                                      media               0.431       0.803  
v2_modernbert_solo_articolo_s1        media               0.428       0.760  
v2_modernbert_solo_articolo_s2        media               0.418       0.753  
v2_modernbert_solo_articolo_s42       media               0.454       0.783  
v2_modernbert_solo_articolo_ensemble3 media               0.431       0.766

In [2]:
rif = ("feature_titolo_tuned", "voto") if ("feature_titolo_tuned", "voto") in T else ("v2_feature_tuned", "voto")
gruppo = pd.read_csv(VERSIONI / "fold_finale.csv").set_index("domain").gruppo
dom = T[rif].index; g = gruppo.reindex(dom).to_numpy(); gruppi = np.unique(g)
pos = {x: np.where(g == x)[0] for x in gruppi}
rng = np.random.default_rng(0)
campioni = [np.concatenate([pos[x] for x in rng.choice(gruppi, len(gruppi))]) for _ in range(B)]


def due(t, idx):
    m = metriche_di(t.y_true.to_numpy()[idx], t.y_pred.to_numpy()[idx]); return m["f1_macro"], m["bin_accuratezza"]


boot = {k: np.array([due(t.reindex(dom), idx) for idx in campioni]) for k, t in T.items()}
iv = {}
for k in T:
    b = boot[k]
    r = {"F1": tutto.loc[k, "f1_macro"], "F1 2.5%": np.percentile(b[:, 0], 2.5), "F1 97.5%": np.percentile(b[:, 0], 97.5),
         "binario": tutto.loc[k, "bin_accuratezza"], "bin 2.5%": np.percentile(b[:, 1], 2.5), "bin 97.5%": np.percentile(b[:, 1], 97.5)}
    if k != rif:
        d = b - boot[rif]
        r.update({"ΔF1": r["F1"] - tutto.loc[rif, "f1_macro"], "ΔF1 2.5%": np.percentile(d[:, 0], 2.5), "ΔF1 97.5%": np.percentile(d[:, 0], 97.5),
                  "P(ΔF1>0)": (d[:, 0] > 0).mean(), "Δbin": r["binario"] - tutto.loc[rif, "bin_accuratezza"],
                  "Δbin 2.5%": np.percentile(d[:, 1], 2.5), "Δbin 97.5%": np.percentile(d[:, 1], 97.5)})
    iv[f"{k[0]} ({k[1]})"] = r
iv = pd.DataFrame(iv).T; iv.to_csv(RESULTS / "03_confronto_intervalli.csv")
print("riferimento:", rif)
iv.round(3)

riferimento: ('feature_titolo_tuned', 'voto')


,F1,F1 2.5%,F1 97.5%,binario,bin 2.5%,bin 97.5%,ΔF1,ΔF1 2.5%,ΔF1 97.5%,P(ΔF1>0),Δbin,Δbin 2.5%,Δbin 97.5%
feature_titolo (voto),0.393,0.316,0.458,0.734,0.670,0.791,0.004,-0.047,0.054,0.542,-0.023,-0.067,0.018
feature_titolo (media),0.375,0.301,0.441,0.737,0.672,0.794,-0.015,-0.077,0.048,0.302,-0.020,-0.068,0.024
feature_titolo_tuned (voto),0.390,0.322,0.452,0.757,0.697,0.813,NaN,NaN,NaN,NaN,NaN,NaN,NaN
feature_titolo_tuned (media),0.369,0.298,0.427,0.724,0.656,0.785,-0.021,-0.056,0.012,0.112,-0.033,-0.074,0.000
v2_feature_tuned (voto),0.370,0.305,0.434,0.773,0.717,0.828,-0.019,-0.067,0.029,0.203,0.016,-0.024,0.060
v2_feature_tuned (media),0.363,0.300,0.420,0.763,0.701,0.822,-0.027,-0.077,0.024,0.159,0.007,-0.033,0.049
v2_modernbert_solo_articolo_s1 (media),0.323,0.266,0.373,0.704,0.631,0.772,-0.067,-0.134,-0.003,0.022,-0.053,-0.112,0.006
v2_modernbert_solo_articolo_s2 (media),0.356,0.291,0.415,0.701,0.634,0.764,-0.034,-0.101,0.032,0.166,-0.056,-0.117,0.004
v2_modernbert_solo_articolo_s42 (media),0.364,0.306,0.416,0.740,0.675,0.802,-0.026,-0.094,0.042,0.240,-0.016,-0.072,0.042
v2_modernbert_solo_articolo_ensemble3 (media),0.341,0.278,0.400,0.707,0.636,0.774,-0.049,-0.116,0.020,0.082,-0.049,-0.108,0.009
